
# Human-directing-robot-dialogue project (HDRD)

* Copyright (c) 2026 Toshiba Europe
* Licensed for research and academic use only.
* Author: Svetlana Stoyanchev 

### Environment: 
* python=3.10 ipykernel, pandas

* This noteboook ran on the outputs generated during IWSDS2026 experiments in output/IWSDS2026_data_w_shortcut
* First  run the experiment run_all_datapoints.sh  generating the output for all data points in output/data_w_shortcut
* change out_dir to output/data_w_shortcut and rerun the notebook for the newly generated scores
* change MODELS to include the list of models_name that ran (out_dir/datapoint/<model_name>)


In [1]:
#instruction version
import json
import pandas as pd
pd.set_option('display.max_rows', 999)
pd.set_option('display.max_columns', 999)
pd.set_option('display.max_colwidth', None)

import os
import json
import traceback
import re

# read LLM-generated output from this directory
#out_dir = '../output/data_w_shortcut'
out_dir = '../output/IWSDS2026_data_w_shortcut'

# read in GT from this directory
in_dir = '../input/data_w_shortcut'

MODELS=['Qwen3-4B-Instruct-2507_temp0_tp0.95', 'gpt-5-mini']

#MODELS=['Qwen3-4B-Instruct-2507_temp0_tp0.95']

# experimental conditions that that were ran in the experiment
# SAME_INTERRUPT refers to 'CAT+RAND' example selection in the paper 
# SIM_100_1_1_1 refers to 'SIM' example selection in the paper
EXP_CONDITIONS=[
  'SAME_INTERRUPT_10_instructOBJS',  # 10 random examples
  'SAME_INTERRUPT_20_instructOBJS',  
  'SAME_INTERRUPT_30_instructOBJS',
  'SAME_INTERRUPT_40_instructOBJS',
  'SAME_INTERRUPT_50_instructOBJS',
  'SAME_INTERRUPT_5_instructOBJS',
  'SIM_100_1_1_1_10_instructOBJS', # 10 similarity-based examples
  'SIM_100_1_1_1_3_instructOBJS',  # 3 similarity-based examples
  'SIM_100_1_1_1_5_instructOBJS']  # 5 similarity-based examples

#EXP_CONDITIONS = ['SIM_100_1_1_1_10_instructOBJS']



### Read in generated results into all_results dictionary

In [2]:

EXP_CONDITIONS_FOUND = set()

all_results = {}
# load all results into a dictionary of the format all_results[example][model][condition] = result
for ex in  os.listdir(out_dir):
    #print (ex)
    if not os.path.isdir(f'{out_dir}/{ex}'):
        continue
    
    all_results[ex] = {}

    for model in MODELS:
        all_results[ex][model] = {}
        for result_fname in os.listdir(f'{out_dir}/{ex}/{model}/'):
            exp_cond = result_fname.replace('.json', '')
            EXP_CONDITIONS_FOUND.add(exp_cond)

            result_path = f'{out_dir}/{ex}/{model}/{result_fname}'
            with open (result_path) as f:
                retstr = f.read().replace('\\"', '"').replace('\\n', '')
                try:
                    if retstr.startswith('"') and  retstr.startswith('"'):
                        #remove quotes in the beginning and end
                        #print( retstr[1:-1])
                        rj = json.loads( retstr[1:-1])
                    else:
                        rj = json.loads( retstr)
                    #print(type(rj))
                except:
                    print(f'Failed to load json for {result_path}')
                    rj = retstr
            
            all_results[ex][model][exp_cond] = {'result': rj}


Failed to load json for ../output/IWSDS2026_data_w_shortcut/118/gpt-5-mini/SIM_100_1_1_1_10_CHEAT_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1235/gpt-5-mini/SIM_100_1_1_1_10_CHEAT_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/936/gpt-5-mini/SAME_INTERRUPT_20_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/226/gpt-5-mini/SIM_100_1_1_1_3_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1055/gpt-5-mini/SAME_INTERRUPT_50_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1055/gpt-5-mini/SAME_INTERRUPT_40_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1055/gpt-5-mini/SAME_INTERRUPT_5_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1055/gpt-5-mini/SAME_INTERRUPT_30_instructOBJS.json
Failed to load json for ../output/IWSDS2026_data_w_shortcut/1055/gpt-5-mini/SAME_INTERRUPT_10_instru

In [3]:

len(all_results), EXP_CONDITIONS

(475,
 ['SAME_INTERRUPT_10_instructOBJS',
  'SAME_INTERRUPT_20_instructOBJS',
  'SAME_INTERRUPT_30_instructOBJS',
  'SAME_INTERRUPT_40_instructOBJS',
  'SAME_INTERRUPT_50_instructOBJS',
  'SAME_INTERRUPT_5_instructOBJS',
  'SIM_100_1_1_1_10_instructOBJS',
  'SIM_100_1_1_1_3_instructOBJS',
  'SIM_100_1_1_1_5_instructOBJS'])

### Read in ground truth into all_results dictionary

In [4]:
for ex in all_results.keys():
    #print (ex)
    with open (f'{in_dir}/{ex}/system_responseGT.json') as f:
        rj = json.load( f) 
        #print(type(rj))   
    all_results[ex]['GT'] = rj

### load df is loaded to get information from it

In [5]:
# read in the state from the csv
#instruction version
import ast

df = pd.read_csv(in_dir + '/data.csv').fillna("")
df['oids_list'] = df['oids'].apply(ast.literal_eval)
len(df)

475

In [6]:
oids_dict = dict(zip(df['utt_id'].astype(str), df['oids_list']))
# map utt_id to context type
contexttype_dict = dict(zip(df['utt_id'].astype(str), df['int_type']))
oids_dict.keys()

dict_keys(['62', '64', '66', '74', '95', '103', '107', '110', '114', '126', '128', '134', '136', '141', '143', '165', '174', '179', '181', '183', '188', '195', '210', '224', '242', '249', '251', '272', '289', '317', '332', '336', '345', '407', '413', '576', '578', '622', '635', '639', '641', '653', '669', '684', '686', '688', '694', '696', '699', '704', '706', '712', '768', '836', '1043', '1071', '1077', '6', '11', '13', '15', '19', '21', '23', '25', '30', '35', '37', '39', '41', '44', '46', '48', '50', '52', '54', '56', '58', '60', '68', '76', '78', '80', '83', '85', '87', '91', '93', '97', '99', '101', '105', '112', '116', '118', '122', '124', '132', '138', '145', '148', '152', '154', '156', '158', '161', '163', '168', '170', '172', '177', '186', '190', '192', '197', '199', '208', '212', '214', '216', '218', '220', '222', '226', '229', '236', '238', '240', '245', '247', '254', '257', '260', '262', '265', '274', '279', '284', '286', '293', '295', '297', '299', '301', '303', '306', '30

### Print all context types and their counts in the data

In [7]:
context_types = dict(df.int_type.value_counts())
context_types

{'IN_CONTEXT_AFTER_FINAL_SUCCESS': np.int64(244),
 'NO_CONTEXT': np.int64(73),
 'IN_CONTEXT_BEFORE_INITIAL': np.int64(50),
 'IN_CONTEXT_AFTER_SUCCESS': np.int64(38),
 'IN_CONTEXT_AFTER_FINAL_FAILED': np.int64(37),
 'IN_CONTEXT_AFTER_FAILED': np.int64(33)}

### Print output for one example, given the ID
* (for the analysis purpose)

In [8]:
def print_output(ex, model, condition):
    
    print(all_results[ex]['GT']['plan'])
    if condition in all_results[ex][model]:
        assert 'result' in all_results[ex][model][condition]
        if 'plan' in all_results[ex][model][condition]['result']:
            print(model, condition, all_results[ex][model][condition]['result']['plan'])
        else:
            print (f'no plan for  {condition} in {model}', all_results[ex][model][condition]['result'])
    else:
        print(f'no {condition} in {model}')

print_output('208', MODELS[0], 'SIM_100_1_1_1_10_instructOBJS')


['Pick_up(mug_1)']
Qwen3-4B-Instruct-2507_temp0_tp0.95 SIM_100_1_1_1_10_instructOBJS ['Go_to(cabinet_1)', 'Open(cabinet_1)', 'Find(mug_1)', 'Pick_up(mug_1)']


In [9]:


def abstractNewObjects(text, objects):
    assert type(text)==str
    # Extract all content inside parentheses
    matches = re.findall(r'\((.*?)\)', text)

    # Split each match by comma and strip whitespace
    objs = [ [item.strip() for item in match.split(',')] for match in matches ]

    objs = []
    for match in matches:
        if ',' in match:
            objs.extend([item.strip() for item in match.split(',')])
        else:
            objs.append(match)

    new_text = text
    for e in objs:
        if '_' in e and e not in objects:
            ot,_ = e.split('_',1)
            new_text = new_text.replace(e, ot + '_XXX')

    return new_text


def abstractNewObjPlan(plan_lst, objects):
    '''for each action of the plan, get objects. If object is NOT in objects list, replace objectID with XXX'''
    assert type(plan_lst)==list
    
    new_plan = []
    for step in plan_lst:
        new_plan.append(abstractNewObjects(step, objects))

    return new_plan


def removeGotoNonFinal(plan):
    assert type (plan )==list
    if len(plan)==0:
        return plan
    
    plan_without_gotos = []
    for action in plan[:-1]:
        if not action.lower().startswith('go_to'):
            plan_without_gotos.append(action)

    plan_without_gotos.append(plan[-1])

    return plan_without_gotos


def comparePredictPlan(p1s,p2s, remove_gotos=False, abstract_new_oids=False, objects = None):
    '''
    adjust both plans:
    1) remove goto
    2) for each object in the plan: if the object is NOT in the current state, change index to XXX to avoid penalizing models that give no-incremental name to the object
    '''
    try:
        if( type(p1s))!=list:
            print (f"correct plan is not a list {p1s}")
        p1ss = [x.replace(" ","").lower() for x in p1s]
        if( type(p2s))!=list:
            print (f"predicted plan is not a list {p2s}")
        p2ss = [x.replace(" ","").lower() for x in p2s]

        if remove_gotos is True: # if goto is NOT the last 
            p1ss = removeGotoNonFinal(p1ss)
            p2ss = removeGotoNonFinal(p2ss)

        if abstract_new_oids is True:
            assert objects is not None
            assert type(objects) == list
            p1ss = abstractNewObjPlan(p1ss, objects)
            p2ss = abstractNewObjPlan(p2ss, objects)


        if p1ss==p2ss:
            return True
        return False
    except:
        print (f"exception in comparePredictPlan p1={p1s}  p2={p2s}", traceback.print_exc()())
        return False

count_exact_match = 0

for ex in  all_results:    
    #print(ex)
    assert 'plan' in all_results[ex]['GT']
    for model in MODELS:
        assert model in all_results[ex]
        for exp_cond in EXP_CONDITIONS:
            if exp_cond not in all_results[ex][model]: # not all models run in all conditions
                continue
            #print(type(all_results[ex][model][exp_cond]['result'] ))
            if type(all_results[ex][model][exp_cond]['result'] )!=dict:
                print ('FORMAT ERROR!!!', ex, model, exp_cond)
                raw_resp = all_results[ex][model][exp_cond]['result'] 
                all_results[ex][model][exp_cond]['result'] = {}
                all_results[ex][model][exp_cond] ['status'] = 'FORMAT_ERROR'
                all_results[ex][model][exp_cond] ['error'] = raw_resp
                continue
            if 'plan' not in all_results[ex][model][exp_cond]['result'] or len(all_results[ex][model][exp_cond]['result']['plan'])==0:
                all_results[ex][model][exp_cond]['status'] = 'PLAN_ERROR'
                continue

            assert 'plan' in all_results[ex][model][exp_cond]['result']


            if comparePredictPlan(all_results[ex][model][exp_cond]['result']['plan'], all_results[ex]['GT']['plan'], remove_gotos=True, 
                                        abstract_new_oids=True, objects=oids_dict[ex]):
                
                if comparePredictPlan(all_results[ex][model][exp_cond]['result']['plan'], all_results[ex]['GT']['plan']):
                    all_results[ex][model][exp_cond]['status']  = 'EXACT_MATCH'           
                else:     
                    all_results[ex][model][exp_cond]['status']  = 'LENIENT_MATCH'    


            else:
                all_results[ex][model][exp_cond]['status']  = 'NO_MATCH'


FORMAT ERROR!!! 936 gpt-5-mini SAME_INTERRUPT_20_instructOBJS
FORMAT ERROR!!! 226 gpt-5-mini SIM_100_1_1_1_3_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_10_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_20_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_30_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_40_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_50_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SAME_INTERRUPT_5_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SIM_100_1_1_1_10_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SIM_100_1_1_1_3_instructOBJS
FORMAT ERROR!!! 1055 gpt-5-mini SIM_100_1_1_1_5_instructOBJS
FORMAT ERROR!!! 728 gpt-5-mini SAME_INTERRUPT_30_instructOBJS
FORMAT ERROR!!! 827 gpt-5-mini SIM_100_1_1_1_10_instructOBJS
FORMAT ERROR!!! 1270 gpt-5-mini SIM_100_1_1_1_5_instructOBJS
FORMAT ERROR!!! 78 gpt-5-mini SAME_INTERRUPT_10_instructOBJS
FORMAT ERROR!!! 78 gpt-5-mini SIM_100_1_1_1_10_instructOBJS
FORMAT ERROR

In [10]:
PRINT_RELATIVE=True

# not exact match errors
def get_count_status(statuses, model, condition, subset=None):
    # subset is a list of context types or None. When None, consider all
    count = 0
    for ex in all_results:
        assert ex in contexttype_dict
        # if subset is defined
        if subset is None or contexttype_dict[ex] in subset:
            if all_results[ex][model][condition]['status'] in statuses:
                count +=1
                #print(f"{ex} {model}:} {json.dumps(all_results[ex]['gpt41']['plan'])}  GT: {json.dumps(all_results[ex]['GT']['plan'])}")
    return count

def get_examples(status, model, condition):
    res = []
    for ex in all_results:
        if all_results[ex][model][condition]['status'] == status:
            res.append(all_results[ex][model][condition]['result'])
            #print(f"{ex} {model}:} {json.dumps(all_results[ex]['gpt41']['plan'])}  GT: {json.dumps(all_results[ex]['GT']['plan'])}")
    return res    


def get_summary():
    count_no_match = 0
    count_exact_match = 0
    count_plan_error = 0
    count_format_error = 0
    for model in MODELS:
        for exp_cond in all_results[ex][model].keys():#EXP_CONDITIONS:
            count_no_match = get_count_status(['NO_MATCH'], model, exp_cond)
            count_lenient_match = get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond)
            count_exact_match = get_count_status(['EXACT_MATCH'], model, exp_cond)
            count_error = get_count_status(['PLAN_ERROR','FORMAT_ERROR'], model, exp_cond)

            print(f"{model}, {exp_cond},  EXACT_MATCH: {count_exact_match}, LENIENT_MATCH: {count_lenient_match}, NO_MATCH: {count_no_match}, PLAN_ERROR:{count_error}"  )


def get_summary_table(cheat=True):
    count_no_match = 0
    count_exact_match = 0
    count_plan_error = 0
    count_format_error = 0
    print('MODEL & CONDITION & TOTAL&  LENIENT\_MATCH & EXACT\_MATCH &  ERROR \\\\')
    for model in MODELS:
        for exp_cond in all_results[ex][model].keys():#EXP_CONDITIONS:
            if (cheat is True and 'CHEAT' in exp_cond) or (cheat is False and 'CHEAT' not in exp_cond):
                count_no_match = get_count_status(['NO_MATCH'], model, exp_cond)
                count_lenient_match = get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond)
                count_exact_match = get_count_status(['EXACT_MATCH'], model, exp_cond)
                count_error = get_count_status(['PLAN_ERROR','FORMAT_ERROR'], model, exp_cond)
                total = count_no_match + + count_lenient_match +  count_error
                if total==0:
                    total=1

                if PRINT_RELATIVE:
                    print(f"{model.replace('_','-')} & {exp_cond.replace('_','-')} & {total} & {count_lenient_match/total:.3f} & {count_exact_match/total:.3f} & {count_error/total:.3f} \\\\\\hline"  )
                else:
                    print(f"{model.replace('_','-')} & {exp_cond.replace('_','-')} & {total} & {count_lenient_match} & {count_exact_match} & {count_error} \\\\\\hline"  )


def get_value_string(num, denum):
    if PRINT_RELATIVE:
        return f"{num/denum:.3f}"
    return f"{num}/{denum}"


def get_lenient_breakdown_table(cheat=True, sep='&'):

    print(f'\n\nMODEL {sep} CONDITION {sep}  ALL {sep} NN {sep} NS {sep} NF{sep} IN {sep} IS {sep}IF \\\\')
    for model in MODELS:
        for exp_cond in EXP_CONDITIONS:
            if (cheat is True and 'CHEAT' in exp_cond) or (cheat is False and 'CHEAT' not in exp_cond):

                print(f"{model.replace('_','-')} & {exp_cond.replace('_','-')} {sep} " +
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond),475)} {sep} " + 
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['NO_CONTEXT']),  context_types['NO_CONTEXT'])} {sep} " +
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['IN_CONTEXT_AFTER_FINAL_SUCCESS']),context_types['IN_CONTEXT_AFTER_FINAL_SUCCESS'])} {sep} " +
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['IN_CONTEXT_AFTER_FINAL_FAILED']),context_types['IN_CONTEXT_AFTER_FINAL_FAILED'])} {sep} " + 
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['IN_CONTEXT_BEFORE_INITIAL']),context_types['IN_CONTEXT_BEFORE_INITIAL'])} {sep} " +
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['IN_CONTEXT_AFTER_SUCCESS']),context_types['IN_CONTEXT_AFTER_SUCCESS'])} {sep}" +
                                                      f"{get_value_string(get_count_status(['LENIENT_MATCH','EXACT_MATCH'], model, exp_cond, ['IN_CONTEXT_AFTER_FAILED']),context_types['IN_CONTEXT_AFTER_FAILED'])}  \\\\\\hline")



# cheat refers to the condition where the test example is present in the few-shot examples
get_lenient_breakdown_table(cheat=False, sep=',')



MODEL , CONDITION ,  ALL , NN , NS , NF, IN , IS ,IF \\
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-10-instructOBJS , 0.476 , 0.466 , 0.529 , 0.568 , 0.320 , 0.474 ,0.242  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-20-instructOBJS , 0.533 , 0.562 , 0.578 , 0.649 , 0.440 , 0.421 ,0.273  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-30-instructOBJS , 0.564 , 0.548 , 0.611 , 0.649 , 0.480 , 0.553 ,0.303  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-40-instructOBJS , 0.583 , 0.575 , 0.627 , 0.622 , 0.600 , 0.553 ,0.242  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-50-instructOBJS , 0.596 , 0.589 , 0.664 , 0.622 , 0.520 , 0.553 ,0.242  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SAME-INTERRUPT-5-instructOBJS , 0.413 , 0.356 , 0.500 , 0.514 , 0.220 , 0.368 ,0.121  \\\hline
Qwen3-4B-Instruct-2507-temp0-tp0.95 & SIM-100-1-1-1-10-instructOBJS , 0.613 , 0.616 , 0.713 , 0.595 , 0.420 , 0.500 ,0.303  \\\hline
Qwen3-